# Movements in Movements Chart

Real Interest Rate vs. Investment (GFCF) for Brazil and peer countries, 2000–2024.

Also known as the "Fan Belt" effect visualization.

In [49]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from plotly.colors import hex_to_rgb

# ── Helpers ──
def lighten(hex_color, factor):
    """Blend hex_color toward white by factor (0=original, 1=white)."""
    r, g, b = hex_to_rgb(hex_color)
    return f'rgb({int(r + (255 - r) * factor)},{int(g + (255 - g) * factor)},{int(b + (255 - b) * factor)})'

def darken(hex_color, factor):
    """Blend hex_color toward black by factor (0=original, 1=black)."""
    r, g, b = hex_to_rgb(hex_color)
    return f'rgb({int(r * (1 - factor))},{int(g * (1 - factor))},{int(b * (1 - factor))})'

def color_gradient(hex_color, n_segments):
    """Return n_segments colors from light→dark for a base hex color."""
    if n_segments <= 1:
        return [hex_color]
    colors = []
    for i in range(n_segments):
        t = i / (n_segments - 1)  # 0→1
        # Go from 55% lightened to 15% darkened
        if t < 0.5:
            colors.append(lighten(hex_color, 0.55 - t * 1.1))
        else:
            colors.append(darken(hex_color, (t - 0.5) * 0.3))
    return colors

# ── Data ──
df_all = pd.read_stata("../Data/Course_Database_Newest.dta")
df_all = df_all.rename(columns={'countrycodeiso': 'countrycode'})

PEERS = ['COL', 'ARG', 'MEX', 'ZAF', 'CHL', 'URY']
PEER_NAMES = {
    'BRA': 'Brazil', 'COL': 'Colombia', 'ARG': 'Argentina',
    'MEX': 'Mexico', 'ZAF': 'South Africa', 'CHL': 'Chile', 'URY': 'Uruguay'
}
PEER_COLORS = {
    'BRA': '#e36209',
    'COL': '#a67b5b', 'ARG': '#a67b5b',
    'MEX': '#7b2d8e', 'ZAF': '#7b2d8e',
    'CHL': '#2c7bb6', 'URY': '#2c7bb6',
}
PEER_SYMBOLS = {
    'BRA': 'circle', 'COL': 'square', 'ARG': 'cross',
    'MEX': 'diamond', 'ZAF': 'x', 'CHL': 'triangle-up', 'URY': 'triangle-down',
}

gfcf_var = 'wdi_ne_gdi_ftot_zs'
rir_var  = 'wdi_fr_inr_rinr'
codes = ['BRA'] + PEERS

sdata = df_all[df_all['countrycode'].isin(codes)].copy()
sdata = sdata[sdata['year'] >= 2000].dropna(subset=[gfcf_var, rir_var])
sdata = sdata.sort_values(['countrycode', 'year'])

# ── Per-country data ──
country_data = {}
for code in codes:
    cdf = sdata[sdata['countrycode'] == code].sort_values('year')
    if cdf.empty:
        continue
    country_data[code] = {
        'xs': cdf[gfcf_var].values,
        'ys': cdf[rir_var].values,
        'yrs': cdf['year'].astype(int).values,
    }

all_xs = sdata[gfcf_var].values
all_ys = sdata[rir_var].values
pad_x = (all_xs.max() - all_xs.min()) * 0.08
pad_y = (all_ys.max() - all_ys.min()) * 0.08
global_xrange = [all_xs.min() - pad_x, all_xs.max() + pad_x]
global_yrange = [all_ys.min() - pad_y, all_ys.max() + pad_y]

valid_codes = [c for c in codes if c in country_data]
N = len(valid_codes)

In [50]:
# ══════════════════════════════════════════════════════════════
# TRACE STRUCTURE per country:
#   [0] scatter markers (always visible)
#   [1] year-label text layer (hidden; toggled by button)
#   [2..2+n_seg-1] gradient line segments (hidden; toggled)
#
# So per country: 2 + n_segments traces
# ══════════════════════════════════════════════════════════════

fig = go.Figure()

# Track trace indices per country for button visibility toggling
trace_map = {}  # code -> {'scatter': idx, 'text': idx, 'segments': [idx, ...]}
trace_idx = 0

for code in valid_codes:
    cd    = country_data[code]
    name  = PEER_NAMES[code]
    color = PEER_COLORS[code]
    sym   = PEER_SYMBOLS[code]
    xs, ys, yrs = cd['xs'], cd['ys'], cd['yrs']
    n_pts = len(xs)
    n_seg = max(n_pts - 1, 1)
    grad  = color_gradient(color, n_seg)

    indices = {'scatter': trace_idx, 'text': None, 'segments': []}

    # ── Scatter markers ──
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers',
        marker=dict(
            color=color,
            size=14 if code == 'BRA' else 11,
            symbol=sym,
            opacity=0.92 if code == 'BRA' else 0.78,
            line=dict(color='white', width=1),
        ),
        name=name,
        showlegend=True,
        hovertemplate=(
            "<b style='font-size:15px'>%{customdata[0]}</b><br>"
            "<span style='font-size:13px'>"
            "Year: <b>%{customdata[1]}</b><br>"
            "GFCF: <b>%{x:.1f}%</b> of GDP<br>"
            "Real Interest Rate: <b>%{y:.1f}%</b>"
            "</span><extra></extra>"
        ),
        customdata=list(zip([name]*n_pts, yrs.astype(str))),
    ))
    trace_idx += 1

    # ── Year-label text overlay ──
    indices['text'] = trace_idx
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        marker=dict(color=color, size=14, symbol=sym,
                    line=dict(color='white', width=1)),
        text=[f"<b>{y}</b>" for y in yrs],
        textposition='top center',
        textfont=dict(size=15, color=color, family='Arial Black'),
        name=name + ' years',
        showlegend=False,
        hoverinfo='skip',
        visible=False,
    ))
    trace_idx += 1

    # ── Gradient line segments (one trace per segment) ──
    for s in range(n_seg):
        seg_color = grad[s]
        indices['segments'].append(trace_idx)
        fig.add_trace(go.Scatter(
            x=[xs[s], xs[s+1]] if s < n_pts - 1 else [xs[s]],
            y=[ys[s], ys[s+1]] if s < n_pts - 1 else [ys[s]],
            mode='lines',
            line=dict(color=seg_color, width=2.5, shape='linear'),
            showlegend=False,
            hoverinfo='skip',
            visible=False,
        ))
        trace_idx += 1

    trace_map[code] = indices

total_traces = trace_idx

In [51]:
# ══════════════════════════════════════════════════════════════
# BUTTONS
# ══════════════════════════════════════════════════════════════
buttons = []

# "Show All": only scatter traces visible
vis_all = [False] * total_traces
for code in valid_codes:
    vis_all[trace_map[code]['scatter']] = True
buttons.append(dict(
    label='  All  ',
    method='update',
    args=[
        {'visible': vis_all},
        {
            'xaxis.range': global_xrange,
            'yaxis.range': global_yrange,
            'annotations': [],
        }
    ],
))

# Per-country buttons
for code in valid_codes:
    cd = country_data[code]
    xs, ys = cd['xs'], cd['ys']
    name  = PEER_NAMES[code]

    vis = [False] * total_traces
    # Show all scatter traces
    for c2 in valid_codes:
        vis[trace_map[c2]['scatter']] = True
    # Show this country's text + gradient segments
    vis[trace_map[code]['text']] = True
    for seg_idx in trace_map[code]['segments']:
        vis[seg_idx] = True

    pad_cx = (xs.max() - xs.min()) * 0.18 + 0.5
    pad_cy = (ys.max() - ys.min()) * 0.18 + 1.0
    xr = [xs.min() - pad_cx, xs.max() + pad_cx]
    yr = [ys.min() - pad_cy, ys.max() + pad_cy]

    buttons.append(dict(
        label=f' {code} ',
        method='update',
        args=[
            {'visible': vis},
            {
                'xaxis.range': xr,
                'yaxis.range': yr,
                'annotations': [],
            },
        ],
    ))

In [52]:
# ══════════════════════════════════════════════════════════════
# LAYOUT (responsive for iframe embedding)
# ══════════════════════════════════════════════════════════════
fig.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=100, b=50),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    xaxis=dict(
        title=dict(
            text='Gross Fixed Capital Formation (% of GDP)',
            font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222'),
            standoff=12,
        ),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        range=global_xrange,
        showgrid=True,
        gridcolor='rgba(180,180,200,0.35)',
        gridwidth=1,
        griddash='dot',
        zeroline=True,
        zerolinecolor='rgba(100,100,120,0.4)',
        zerolinewidth=1.5,
        showline=True,
        linecolor='#444',
        linewidth=1.5,
        mirror=False,
        ticks='outside',
        ticklen=5,
        tickcolor='#888',
        minor=dict(showgrid=True, gridcolor='rgba(200,200,210,0.2)', griddash='dot'),
    ),
    yaxis=dict(
        title=dict(
            text='Real Interest Rate (%)',
            font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222'),
            standoff=12,
        ),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        range=global_yrange,
        showgrid=True,
        gridcolor='rgba(180,180,200,0.35)',
        gridwidth=1,
        griddash='dot',
        zeroline=True,
        zerolinecolor='rgba(220,60,60,0.35)',
        zerolinewidth=2,
        showline=True,
        linecolor='#444',
        linewidth=1.5,
        mirror=False,
        ticks='outside',
        ticklen=5,
        tickcolor='#888',
        minor=dict(showgrid=True, gridcolor='rgba(200,200,210,0.2)', griddash='dot'),
    ),
    title=dict(
        text=(
            '<b>The "Fan Belt" Effect</b>'
            '<br><span style="font-size:17px;color:#555">'
            'Real Interest Rate vs. Investment, 2000–2024</span>'
        ),
        font=dict(size=23, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        y=0.96,
    ),
    legend=dict(
        orientation='v',
        yanchor='top', y=0.98,
        xanchor='right', x=0.99,
        font=dict(size=14, family='Roboto Condensed, sans-serif'),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(200,200,210,0.4)',
        borderwidth=1,
        itemsizing='constant',
        tracegroupgap=4,
    ),
    hovermode='closest',
    hoverlabel=dict(
        font_size=16,
        font_family='Roboto Condensed, sans-serif',
        font_color='black',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#bbb',
    ),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, xanchor='center',
        y=1.0, yanchor='bottom',
        bgcolor='rgba(245,245,250,0.95)',
        bordercolor='rgba(180,180,200,0.6)',
        borderwidth=1,
        font=dict(size=15, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        buttons=buttons,
        pad=dict(r=6, t=6, b=6, l=6),
    )],
    annotations=[],
)

fig.show()

# Save as HTML - strip inline dimensions and use JS to resize
import re

plot_div = fig.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})

# Remove the inline width/height style from the plotly div
plot_div = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div)

html_content = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div}
<script>
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            var w = window.innerWidth;
            var h = window.innerHeight;
            Plotly.relayout(gd, {{
                width: w,
                height: h
            }});
        }}
    }}
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(resizePlot, 50);
        setTimeout(resizePlot, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("mov-in-mov.html", "w", encoding="utf-8") as f:
    f.write(html_content)
    
print("HTML saved. Chart will resize to fill iframe.")

HTML saved. Chart will resize to fill iframe.


# Manufacturing Value Added as Share of GDP

Premature deindustrialisation visualization for Brazil and peer countries, 2000–present.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

# ── Data ──
df_manuf = pd.read_stata("../Data/Course_Database_Newest.dta")
df_manuf = df_manuf.rename(columns={'countrycodeiso': 'countrycode'})

manuf_var = 'wdi_nv_ind_manf_zs'
PEERS_M = ['COL', 'ARG', 'MEX', 'ZAF', 'CHL', 'URY']
all_codes_m = ['BRA'] + PEERS_M

PEER_NAMES_M = {
    'BRA': 'Brazil', 'COL': 'Colombia', 'ARG': 'Argentina',
    'MEX': 'Mexico', 'ZAF': 'South Africa', 'CHL': 'Chile', 'URY': 'Uruguay'
}
PEER_COLORS_M = {
    'BRA': '#e36209',
    'COL': '#a67b5b', 'ARG': '#a67b5b',
    'MEX': '#7b2d8e', 'ZAF': '#7b2d8e',
    'CHL': '#2c7bb6', 'URY': '#2c7bb6',
}
PEER_SYMBOLS_M = {
    'BRA': 'circle', 'COL': 'square', 'ARG': 'cross',
    'MEX': 'diamond', 'ZAF': 'x', 'CHL': 'triangle-up', 'URY': 'triangle-down',
}
sdata_m = df_manuf[df_manuf['countrycode'].isin(all_codes_m) & (df_manuf['year'] >= 2000)].copy()
sdata_m = sdata_m.dropna(subset=[manuf_var]).sort_values(['countrycode', 'year'])

fig_manuf = go.Figure()

for code in all_codes_m:
    cdf = sdata_m[sdata_m['countrycode'] == code].sort_values('year')
    if cdf.empty:
        continue
    name = PEER_NAMES_M[code]
    color = PEER_COLORS_M[code]
    sym = PEER_SYMBOLS_M[code]
    lw = 3.5 if code == 'BRA' else 1.8
    sz = 14 if code == 'BRA' else 10

    fig_manuf.add_trace(go.Scatter(
        x=cdf['year'], y=cdf[manuf_var],
        mode='lines+markers',
        line=dict(color=color, width=lw),
        marker=dict(
            color=color, size=sz, symbol=sym,
            line=dict(color='white', width=0.8),
        ),
        name=name,
        hovertemplate=(
            f"<b style='font-size:15px'>{name}</b><br>"
            "<span style='font-size:13px'>"
            "Year: <b>%{x}</b><br>"
            "Manufacturing VA: <b>%{y:.1f}%</b> of GDP"
            "</span><extra></extra>"
        ),
    ))

# ── Layout (responsive for iframe embedding) ──
fig_manuf.update_layout(
    template='plotly_white',
    autosize=True,
    margin=dict(l=60, r=30, t=100, b=50),
    plot_bgcolor='rgba(252,252,255,1)',
    paper_bgcolor='white',
    font=dict(family='Roboto Condensed, sans-serif', color='#1a1a2e'),
    title=dict(
        text=(
            '<b>Manufacturing Value Added as Share of GDP</b>'
            '<br><span style="font-size:17px;color:#555">'
            'Premature Deindustrialisation, 2000–present</span>'
        ),
        font=dict(size=23, family='Roboto Condensed, sans-serif', color='#1a1a2e'),
        x=0.5, xanchor='center',
        y=0.96,
    ),
    xaxis=dict(
        title=dict(text='Year', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
    ),
    yaxis=dict(
        title=dict(text='Manufacturing VA (% of GDP)', font=dict(size=19, family='Roboto Condensed, sans-serif', color='#222')),
        tickfont=dict(size=16, color='#333'),
        ticksuffix='%',
        showgrid=True, gridcolor='rgba(180,180,200,0.35)', griddash='dot',
        showline=True, linecolor='#444', linewidth=1.5,
        ticks='outside', ticklen=5, tickcolor='#888',
    ),
    legend=dict(
        orientation='v',
        yanchor='top', y=0.98,
        xanchor='right', x=0.99,
        font=dict(size=14, family='Roboto Condensed, sans-serif'),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(200,200,210,0.4)',
        borderwidth=1,
        itemsizing='constant',
    ),
    hovermode='x unified',
    hoverlabel=dict(
        font_size=16,
        font_family='Roboto Condensed, sans-serif',
        font_color='black',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#bbb',
    ),
    annotations=[],
)

fig_manuf.show()

# Save as HTML with responsive resizing
plot_div_m = fig_manuf.to_html(include_plotlyjs=True, full_html=False, config={'responsive': True})
plot_div_m = re.sub(r'style="height:\s*\d+px;\s*width:\s*\d+px;"', 'style="width:100%; height:100%;"', plot_div_m)

html_content_m = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{
            width: 100%;
            height: 100%;
            overflow: hidden;
        }}
        .js-plotly-plot {{
            width: 100% !important;
            height: 100% !important;
        }}
    </style>
</head>
<body>
{plot_div_m}
<script>
    function resizePlot() {{
        var gd = document.querySelector('.js-plotly-plot');
        if (gd && window.Plotly) {{
            var w = window.innerWidth;
            var h = window.innerHeight;
            Plotly.relayout(gd, {{
                width: w,
                height: h
            }});
        }}
    }}
    window.addEventListener('load', function() {{
        resizePlot();
        setTimeout(resizePlot, 50);
        setTimeout(resizePlot, 200);
    }});
    window.addEventListener('resize', resizePlot);
</script>
</body>
</html>
'''

with open("manuf-va.html", "w", encoding="utf-8") as f:
    f.write(html_content_m)
    
print("Manufacturing VA chart saved to manuf-va.html")